# 1. Load the Cleaned Data

In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\processed"
FINAL     = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\final"

df = pd.read_csv(os.path.join(PROCESSED, "daedalus_merged.csv"))
print(f"Loaded: {df.shape}")
print(df.columns.tolist())

Loaded: (333, 18)
['country', 'city', 'aqi_value', 'aqi_category', 'pm2.5_aqi_value', 'meal_cheap_usd', 'meal_midrange_usd', 'rent_1br_outside_usd', 'rent_3br_city_usd', 'basic_utilities_usd', 'monthly_pass_usd', 'avg_net_salary_usd', 'quality_of_life_index', 'popcity', 'densitycity', 'congestion2019', 'crime_index', 'safety_index']


# 2. Min-max normalizer (0 to 100, higher = better):


In [2]:
def normalize(series, invert=False):
    """
    Scales a series to 0-100.
    invert=True when lower raw value = better score (e.g. crime, AQI, rent)
    """
    mn, mx = series.min(), series.max()
    scaled = (series - mn) / (mx - mn) * 100
    if invert:
        scaled = 100 - scaled
    return scaled.round(2)

# 3. Pillar 1: Air Quality Score

In [3]:
# Lower AQI = better air = higher score → invert both
df['aqi_score']    = normalize(df['aqi_value'],       invert=True)
df['pm25_score']   = normalize(df['pm2.5_aqi_value'], invert=True)

# Weighted average within pillar (PM2.5 is more health-relevant)
df['air_quality_score'] = (df['aqi_score'] * 0.4 + df['pm25_score'] * 0.6).round(2)

print("Air quality score sample:")
print(df[['city', 'aqi_value', 'pm2.5_aqi_value', 'air_quality_score']]\
      .sort_values('air_quality_score', ascending=False).head(10))

Air quality score sample:
           city  aqi_value  pm2.5_aqi_value  air_quality_score
145    brasilia         21               11              99.18
72     canberra         24               11              98.93
325     uppsala         29                9              98.79
29    stockholm         31                8              98.76
328       turku         36                5              98.75
159        cork         27               13              98.41
51      tampere         30               12              98.29
109    limerick         23               17              98.20
151  wellington         24               17              98.12
253   stavanger         29               14              98.11


# 4. Pillar 2a: Absolute cost score (lower cost = better):

In [4]:
# Create a composite cost index first
# Weighted sum of key cost indicators
df['cost_index_raw'] = (
    df['meal_cheap_usd']       * 0.20 +
    df['rent_1br_outside_usd'] * 0.50 +
    df['basic_utilities_usd']  * 0.20 +
    df['monthly_pass_usd']     * 0.10
)

# Lower cost = better score → invert
df['cost_score_absolute'] = normalize(df['cost_index_raw'], invert=True).round(2)

print("Absolute cost score (lower cost cities rank higher):")
print(df[['city', 'country', 'cost_index_raw', 'cost_score_absolute']]\
      .sort_values('cost_score_absolute', ascending=False).head(10))

Absolute cost score (lower cost cities rank higher):
              city                      country  cost_index_raw  \
1    dar es salaam  United Republic of Tanzania        1321.665   
103      cartagena                        Spain        2983.230   
79           wuhan                        China        3190.215   
162        qingdao                        China        3328.443   
293         hobart                    Australia        3365.823   
185          odesa                      Ukraine        3416.002   
177      chongqing                        China        3427.270   
121        catania                        Italy        3471.944   
198         bogota                     Colombia        3505.352   
62           tokyo                        Japan        3512.745   

     cost_score_absolute  
1                 100.00  
103                91.09  
79                 89.98  
162                89.24  
293                89.04  
185                88.77  
177                8

# 5.  Pillar 2b: Affordability ratio score (cost vs local salary):

In [5]:
# Affordability = what % of monthly salary does basic living cost?
# Lower % = more affordable relative to local wages
df['monthly_living_cost'] = (
    df['meal_cheap_usd']       * 30  +   # ~30 cheap meals/month
    df['rent_1br_outside_usd']       +   # monthly rent
    df['basic_utilities_usd']        +   # utilities
    df['monthly_pass_usd']               # transport
)

df['affordability_ratio'] = (df['monthly_living_cost'] / df['avg_net_salary_usd'] * 100).round(2)

# Lower ratio = more affordable → invert
df['cost_score_affordability'] = normalize(df['affordability_ratio'], invert=True).round(2)

print("Affordability ratio score (cost relative to local salary):")
print(df[['city', 'country', 'monthly_living_cost', 'avg_net_salary_usd',
          'affordability_ratio', 'cost_score_affordability']]\
      .sort_values('cost_score_affordability', ascending=False).head(10))

Affordability ratio score (cost relative to local salary):
            city                   country  monthly_living_cost  \
59      new york                   Unknown             24569.74   
163  jersey city  United States of America             19170.17   
327       boston  United States of America             23046.39   
294   washington  United States of America             22242.23   
17   los angeles  United States of America             23547.68   
14     new haven  United States of America             20498.08   
108        miami  United States of America             24922.94   
33      san jose                   Unknown             23188.43   
68     san diego                Costa Rica             23621.40   
193     hamilton  United States of America             36964.75   

     avg_net_salary_usd  affordability_ratio  cost_score_affordability  
59              7146.84               343.78                    100.00  
163             4500.00               426.00             

# 6. Pillar 3: Safety score:

In [6]:
# Safety index is already 0-100 (higher = safer), just normalize range
# Crime index is inverse (higher = more crime)
df['safety_score'] = (
    normalize(df['safety_index'], invert=False) * 0.6 +
    normalize(df['crime_index'],  invert=True)  * 0.4
).round(2)

print("Safety score sample:")
print(df[['city', 'country', 'crime_index', 'safety_index', 'safety_score']]\
      .sort_values('safety_score', ascending=False).head(10))

Safety score sample:
          city               country  crime_index  safety_index  safety_score
56   abu dhabi  United Arab Emirates         11.2          88.8        100.00
314       doha                 Qatar         14.5          85.5         95.44
267     taipei                 China         15.1          84.9         94.61
199    sharjah  United Arab Emirates         15.9          84.1         93.51
22       dubai  United Arab Emirates         16.4          83.6         92.82
39        bern           Switzerland         17.7          82.3         91.02
231     zurich           Switzerland         18.3          81.7         90.19
107     munich               Germany         18.8          81.2         89.50
48   trondheim                Norway         19.1          80.9         89.09
23       basel           Switzerland         21.6          78.4         85.64


# 7. Pillar 4: Urban density score:

In [7]:
# Density is nuanced — very high AND very low density are both bad
# Sweet spot is moderate density (walkable but not overcrowded)
# We use a "moderate density preference" curve instead of simple invert

density_median = df['densitycity'].median()
density_std    = df['densitycity'].std()

# Score peaks at median density, falls off on both sides
df['density_score_raw'] = np.exp(
    -0.5 * ((df['densitycity'] - density_median) / density_std) ** 2
) * 100

# Congestion — lower is better
df['congestion_score'] = normalize(df['congestion2019'], invert=True)

# Combined urban score
df['urban_score'] = (
    df['density_score_raw'] * 0.5 +
    df['congestion_score']  * 0.5
).round(2)

print("Urban score sample:")
print(df[['city', 'densitycity', 'congestion2019', 'urban_score']]\
      .sort_values('urban_score', ascending=False).head(10))

Urban score sample:
            city  densitycity  congestion2019  urban_score
307     syracuse  2236.719840             8.0        98.93
137   greensboro   776.646573             7.0        98.30
218    milwaukee  2387.774876             9.0        97.84
244    cleveland  1970.928904             9.0        97.82
128    worcester  1870.957205             9.0        97.78
66   minneapolis  2735.464988             9.0        97.63
6       richmond  1315.768898             9.0        97.20
246  little rock   622.429782             8.0        96.85
296    rochester  2273.068299            10.0        96.81
269      buffalo  2498.720160            10.0        96.74


# 8. Final composite livability score:

In [16]:
# Default weights — user can change these in the Streamlit app
WEIGHTS = {
    'air_quality': 0.25,
    'cost':        0.30,
    'safety':      0.25,
    'urban':       0.20,
}

# Absolute cost version
df['livability_score_absolute'] = (
    df['air_quality_score']    * WEIGHTS['air_quality'] +
    df['cost_score_absolute']  * WEIGHTS['cost']        +
    df['safety_score']         * WEIGHTS['safety']      +
    df['urban_score']          * WEIGHTS['urban']
).round(2)

# Affordability version
df['livability_score_affordability'] = (
    df['air_quality_score']         * WEIGHTS['air_quality'] +
    df['cost_score_affordability']  * WEIGHTS['cost']        +
    df['safety_score']              * WEIGHTS['safety']      +
    df['urban_score']               * WEIGHTS['urban']
).round(2)

# Rank both
df['rank_absolute']      = df['livability_score_absolute'].rank(ascending=False).astype(int)
df['rank_affordability'] = df['livability_score_affordability'].rank(ascending=False).astype(int)

print("=" * 60)
print("TOP 20 — Absolute cost model")
print("=" * 60)
print(df[['rank_absolute', 'city', 'country', 'livability_score_absolute']]\
      .sort_values('rank_absolute').head(20).to_string(index=False))

print("\n")
print("=" * 60)
print("TOP 20 — Affordability model")
print("=" * 60)
print(df[['rank_affordability', 'city', 'country', 'livability_score_affordability']]\
      .sort_values('rank_affordability').head(20).to_string(index=False))

TOP 20 — Absolute cost model
 rank_absolute      city      country  livability_score_absolute
             1  canberra    Australia                      90.00
             2 ljubljana     Slovenia                      87.61
             3   tampere      Finland                      87.18
             4 stavanger       Norway                      85.48
             5    munich      Germany                      85.40
             6    malaga        Spain                      85.08
             7 trondheim       Norway                      84.81
             8 eindhoven  Netherlands                      84.57
             9    hobart    Australia                      84.23
            10      graz      Austria                      84.13
            11 groningen  Netherlands                      83.83
            12  adelaide    Australia                      83.78
            13    ottawa       Canada                      83.70
            14      doha        Qatar                      83

In [10]:
# Build a city -> country lookup from the most reliable source
country_lookup = pd.read_csv(r"C:\Users\alokhande\Desktop\Project_Daedalus\data\raw\crime-index-2023.csv")
country_lookup.columns = country_lookup.columns.str.lower()
country_lookup['city'] = country_lookup['city'].str.strip().str.lower()
country_lookup = country_lookup[['city', 'country']].drop_duplicates(subset=['city'])

# Also add from AQI as backup
aqi_lookup = pd.read_csv(r"C:\Users\alokhande\Desktop\Project_Daedalus\data\raw\global air pollution dataset.csv")
aqi_lookup.columns = aqi_lookup.columns.str.lower()
aqi_lookup['city'] = aqi_lookup['city'].str.strip().str.lower()
aqi_lookup = aqi_lookup[['city', 'country']].drop_duplicates(subset=['city'])

# Combine both lookups — crime first, AQI as fallback
full_lookup = pd.concat([country_lookup, aqi_lookup]).drop_duplicates(subset=['city'])

# Apply to scored dataset
df = df.drop(columns=['country'])
df = df.merge(full_lookup, on='city', how='left')
df['country'] = df['country'].fillna('Unknown')

# Verify fixes
checks = ['stavanger', 'ottawa', 'vienna', 'bergen']
print(df[df['city'].isin(checks)][['city', 'country']])

          city   country
80      bergen    Norway
217     ottawa    Canada
253  stavanger    Norway
329     vienna   Austria


In [13]:
# Fix US state codes — replace any 2-letter uppercase country with USA
import re

def fix_country(country):
    if pd.isna(country):
        return 'Unknown'
    # if it's a 2-letter code it's a US state abbreviation
    if re.match(r'^[A-Z]{2}$', str(country).strip()):
        return 'United States of America'
    return country

df['country'] = df['country'].apply(fix_country)

# Verify
checks = ['salt lake city', 'raleigh', 'canberra', 'vienna', 'munich']
print(df[df['city'].isin(checks)][['city', 'country']])

               city                   country
72         canberra                 Australia
107          munich                   Germany
130  salt lake city  United States of America
239         raleigh  United States of America
329          vienna                   Austria


# 9. Save 

In [15]:
output_path = os.path.join(FINAL, "daedalus_scored.csv")
df.to_csv(output_path, index=False)
print(f"Saved → {output_path}")
print(f"Shape: {df.shape}")
print(f"\nFinal columns:\n{df.columns.tolist()}")

Saved → C:\Users\alokhande\Desktop\Project_Daedalus\data\final\daedalus_scored.csv
Shape: (333, 34)

Final columns:
['city', 'aqi_value', 'aqi_category', 'pm2.5_aqi_value', 'meal_cheap_usd', 'meal_midrange_usd', 'rent_1br_outside_usd', 'rent_3br_city_usd', 'basic_utilities_usd', 'monthly_pass_usd', 'avg_net_salary_usd', 'quality_of_life_index', 'popcity', 'densitycity', 'congestion2019', 'crime_index', 'safety_index', 'aqi_score', 'pm25_score', 'air_quality_score', 'cost_index_raw', 'cost_score_absolute', 'monthly_living_cost', 'affordability_ratio', 'cost_score_affordability', 'safety_score', 'density_score_raw', 'congestion_score', 'urban_score', 'livability_score_absolute', 'livability_score_affordability', 'rank_absolute', 'rank_affordability', 'country']
